In [1]:
# -*- coding: utf-8 -*-
import os
import random
import numpy as np
import pandas as pd
import optuna
from dataclasses import dataclass
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from xgboost import XGBClassifier

# =====================================================
# CONFIG
# =====================================================
@dataclass
class Config:
    data_dir: str = "/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad"
    max_weeks: int = 40
    test_size: float = 0.2
    random_state: int = 42
    n_trials: int = 20  # Số lần Optuna chạy thử (tăng lên 50 nếu có thời gian)

CFG = Config()
EARLY_MARKERS = [0.1, 0.4, 0.6, 0.8, 1.0]
# EARLY_MARKERS = [0.2]


# =====================================================
# UTILS (Giữ nguyên logic xử lý dữ liệu của bạn)
# =====================================================
def map_day_to_week(day, max_weeks):
    if pd.isna(day) or day < 0: return 0
    return min(int(day // 7), max_weeks - 1)

def load_data(data_dir):
    files = {"student_info": "studentInfo.csv", "student_vle": "studentVle.csv", 
             "student_assessment": "studentAssessment.csv", "assessment": "assessments.csv", "vle": "vle.csv"}
    data = {}
    for k, v in files.items():
        path = os.path.join(data_dir, v)
        data[k] = pd.read_csv(path)
    return data

# =====================================================
# TARGET
# =====================================================

def build_target(df):

    df = df.copy()

    df["target"] = df["final_result"].apply(
        lambda x: 1 if x in ["Pass", "Distinction"] else 0
    )

    return df

# =====================================================
# CLICK SERIES
# =====================================================

def build_click_series(df_student_vle, max_weeks, cutoff_week):

    df = df_student_vle.copy()

    df["week_idx"] = df["date"].apply(
        lambda x: map_day_to_week(x, max_weeks)
    )
    df = df[df["week_idx"] < cutoff_week]

    grouped = df.groupby(
        ["id_student", "code_module", "code_presentation", "week_idx"]
    )["sum_click"].sum().reset_index()

    pivot = grouped.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="week_idx",
        values="sum_click",
        fill_value=0
    )

    pivot = pivot.reindex(columns=list(range(max_weeks)), fill_value=0)

    pivot.columns = [f"click_wk_{i}" for i in pivot.columns]

    return pivot.reset_index()

# =====================================================
# ACTIVITY FEATURES
# =====================================================

def build_activity_features(student_vle, vle, max_weeks, cutoff_week):

    # merge activity type
    df = student_vle.merge(
        vle[["id_site", "activity_type"]],
        on="id_site",
        how="left"
    )

    # map day -> week
    df["week_idx"] = df["date"].apply(
        lambda x: map_day_to_week(x, max_weeks)
    )
    df = df[df["week_idx"] < cutoff_week]

    # aggregate clicks per activity type per week
    activity = df.pivot_table(
        index=["id_student","code_module","code_presentation","week_idx"],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    # =====================================================
    # get all activity types dynamically
    # =====================================================

    activity_types = vle["activity_type"].unique().tolist()

    # ensure all activity columns exist
    for col in activity_types:
        if col not in activity.columns:
            activity[col] = 0

    # =====================================================
    # TOTAL CLICKS
    # =====================================================

    activity["total"] = activity[activity_types].sum(axis=1)

    # inactive feature
    activity["inactive"] = (activity["total"] == 0).astype(int)

    # avoid divide-by-zero
    activity["total"] = activity["total"].replace(0,1)

    # =====================================================
    # RATIOS
    # =====================================================

    for col in activity_types:
        activity[f"{col}_ratio"] = activity[col] / activity["total"]

    ratio_cols = [f"{col}_ratio" for col in activity_types]

    # =====================================================
    # ENTROPY
    # =====================================================

    def entropy(row):

        vals = row[activity_types].values.astype(float)

        total = vals.sum()

        if total == 0:
            return np.nan

        p = vals / total
        p = p[p > 0]

        return -(p * np.log(p)).sum()

    activity["entropy"] = activity.apply(entropy, axis=1)

    # =====================================================
    # FEATURES USED IN MODEL
    # =====================================================

    feature_cols = ratio_cols + ["entropy", "inactive"]

    # =====================================================
    # WEEKLY PIVOT
    # =====================================================

    pivot = activity.pivot_table(
        index=["id_student","code_module","code_presentation"],
        columns="week_idx",
        values=feature_cols,
        fill_value=0
    )

    # =====================================================
    # ENSURE FULL WEEK RANGE
    # =====================================================

    full_cols = pd.MultiIndex.from_product(
        [
            feature_cols,
            range(max_weeks)
        ]
    )

    pivot = pivot.reindex(columns=full_cols, fill_value=0)

    # =====================================================
    # FLATTEN COLUMN NAMES
    # =====================================================

    pivot.columns = [
        f"{feat}_wk_{week}"
        for feat, week in pivot.columns
    ]

    # fill entropy NaN
    entropy_cols = [c for c in pivot.columns if "entropy" in c]
    pivot[entropy_cols] = pivot[entropy_cols].fillna(0)

    return pivot.reset_index()

def build_assessment_series_safe(df_student_assessment, df_assessment, max_weeks, cutoff_week):
    """
    Args:
        df_student_assessment: Bảng kết quả nộp bài của sinh viên.
        df_assessment: Bảng thông tin các bài kiểm tra (có cột 'date' là hạn chót).
        max_weeks: Tổng số tuần tối đa của khóa học.
        cutoff_week: Tuần hiện tại (chỉ lấy dữ liệu trước tuần này để dự báo).
    """

    # 1. Kết hợp dữ liệu
    merged = df_student_assessment.merge(df_assessment, on="id_assessment")

    # 2. XÁC ĐỊNH TUẦN DỰA TRÊN HẠN CHÓT
    # Giả sử 1 tuần có 7 ngày. Bài kiểm tra có 'date' rơi vào ngày nào thì thuộc tuần đó.
    merged["date"] = merged["date"].fillna(0)
    merged["week_idx"] = (merged["date"] // 7).astype(int)

    # 3. CHẶN DỮ LIỆU TƯƠNG LAI
    # Chỉ giữ lại các bài kiểm tra có hạn chót nằm trong khoảng từ tuần 0 đến cutoff_week
    merged = merged[merged["week_idx"] < cutoff_week].copy()

    # 4. TÍNH TOÁN LATENESS AN TOÀN
    # Nếu chưa nộp (score NaN), ta coi là trễ so với mốc cutoff hoặc ngày hạn chót
    def calc_lateness(row):
        if pd.isna(row["date_submitted"]):
            # Nếu chưa nộp bài đã quá hạn, tính độ trễ tới thời điểm hiện tại (cutoff)
            cutoff_day = cutoff_week * 7
            return max(0, cutoff_day - row["date"])
        else:
            # Nếu đã nộp, tính độ trễ thực tế (có thể âm nếu nộp sớm)
            return row["date_submitted"] - row["date"]

    merged["lateness"] = merged.apply(calc_lateness, axis=1)
    merged["score"] = merged["score"].fillna(0)
    merged["weight"] = merged["weight"].fillna(0)

    # 5. TẠO BẢNG XOAY (Pivot Tables)
    idx = ["id_student", "code_module", "code_presentation"]
    # Chỉ tạo các cột từ tuần 0 đến cutoff_week - 1
    target_weeks = list(range(cutoff_week))

    # Điểm số trung bình theo tuần
    score_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="score", aggfunc="mean"
    ).reindex(columns=target_weeks, fill_value=0)

    # Độ trễ trung bình theo tuần
    late_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="lateness", aggfunc="mean"
    ).reindex(columns=target_weeks, fill_value=0)

    # Trọng số bài thi trong tuần
    weight_pivot = merged.pivot_table(
        index=idx, columns="week_idx", values="weight", aggfunc="max"
    ).reindex(columns=target_weeks, fill_value=0)

    # 6. ĐỔI TÊN CỘT ĐỂ PHÂN BIỆT
    score_pivot.columns = [f"score_wk_{i}" for i in score_pivot.columns]
    late_pivot.columns = [f"late_wk_{i}" for i in late_pivot.columns]
    weight_pivot.columns = [f"weight_wk_{i}" for i in weight_pivot.columns]

    # 7. GỘP LẠI THÀNH DATAFRAME CUỐI CÙNG
    df_final = score_pivot.join(late_pivot).join(weight_pivot).reset_index()

    return df_final
    

def build_dataset(data, cutoff_week):
    # Sử dụng các hàm build của bạn để tạo dataframe tổng hợp
    df_info = data["student_info"].copy()
    df_info["target"] = df_info["final_result"].apply(lambda x: 1 if x in ["Pass", "Distinction"] else 0)
    
    # Giả sử các hàm build_..._series đã được định nghĩa phía trên
    df_click = build_click_series(data["student_vle"], CFG.max_weeks, cutoff_week)
    df_assess = build_assessment_series_safe(data["student_assessment"], data["assessment"], CFG.max_weeks, cutoff_week)
    df_activity = build_activity_features(data["student_vle"], data["vle"], CFG.max_weeks, cutoff_week)

    keys = ["id_student", "code_module", "code_presentation"]
    df = df_info.merge(df_click, on=keys, how="left").merge(df_assess, on=keys, how="left").merge(df_activity, on=keys, how="left")
    return df.fillna(0)

def prepare_features(df, activity_types, cutoff_week):
    # Logic tách temporal và static (Giữ nguyên của bạn)
    all_cols = df.columns.tolist()
    temporal_cols = [c for c in all_cols if "_wk_" in c and int(c.split("_wk_")[-1]) < cutoff_week]
    
    exclude = ["id_student", "code_module", "code_presentation", "final_result", "target"] + [c for c in all_cols if "_wk_" in c]
    obj_cols = [c for c in df.select_dtypes(include=["object"]).columns if c not in exclude]
    df_proc = pd.get_dummies(df, columns=obj_cols, drop_first=True)
    
    static_cols = [c for c in df_proc.columns if c not in exclude and "_wk_" not in c]
    X_static = df_proc[static_cols].astype(np.float32).values
    
    # Flatten temporal
    temporal_data = []
    for i in range(cutoff_week):
        week_data = df[[c for c in temporal_cols if f"_wk_{i}" in c]].values
        temporal_data.append(week_data)
    X_temporal = np.concatenate(temporal_data, axis=1) if temporal_data else np.array([]).reshape(len(df), 0)
    
    X = np.concatenate([X_static, X_temporal], axis=1)
    y = df["target"].values
    return X, y


def tune_xgb_f1_cv(X_train, y_train, n_trials=20):
    """Optuna Tuning tham số XGBoost với Threshold mặc định 0.5 dùng Cross-Validation."""
    def objective(trial):
        params = {
            "n_estimators": 1000, # Tăng lên để model đủ sức học
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            # Thêm các tham số chính quy hóa để tránh overfit và tăng Accuracy
            "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
            "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "random_state": 42,
            "tree_method": "hist"
        }

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        oof_f1_scores = []

        for train_idx, val_idx in skf.split(X_train, y_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            # Tính toán scale_pos_weight để cân bằng lớp dữ liệu tự động
            scale_pos = (y_tr == 0).sum() / (y_tr == 1).sum()
            
            model = XGBClassifier(**params, scale_pos_weight=scale_pos, early_stopping_rounds=50)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            
            # Dự đoán với Threshold mặc định 0.5
            preds = model.predict(X_val) 
            oof_f1_scores.append(f1_score(y_val, preds))

        return np.mean(oof_f1_scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    return study.best_params

# =====================================================
# MAIN RUNNER
# =====================================================
def run_xgb_tuned():
    print("Loading data...")
    data = load_data(CFG.data_dir)
    results = []

    for marker in EARLY_MARKERS:
        print(f"\n===== CHẠY EARLY PREDICTION: {marker*100:.0f}% MỐC THỜI GIAN =====")
        cutoff_week = int(CFG.max_weeks * marker)
        
        df = build_dataset(data, cutoff_week)
        activity_types = data["vle"]["activity_type"].unique().tolist()
        X, y = prepare_features(df, activity_types, cutoff_week)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=CFG.test_size, stratify=y, random_state=CFG.random_state
        )

        # 1. Tuning tham số với Threshold cố định 0.5
        print(f"Bắt đầu Tuning Optuna (Tối ưu F1 với Thr=0.5)...")
        best_params = tune_xgb_f1_cv(X_train, y_train, n_trials=CFG.n_trials)

        # 2. Huấn luyện Model cuối cùng
        print(f"Huấn luyện model cuối với bộ tham số tốt nhất...")
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        final_model = XGBClassifier(**best_params, scale_pos_weight=scale_pos_weight, n_estimators=1000)
        final_model.fit(X_train, y_train)

        # 3. Đánh giá trên tập Test
        y_pred_test = final_model.predict(X_test)
        test_f1 = f1_score(y_test, y_pred_test)
        test_acc = accuracy_score(y_test, y_pred_test)

        print(f"KẾT QUẢ {marker*100:.0f}%: Test F1 = {test_f1:.5f} | Test Acc = {test_acc:.5f}")
        # In report với 5 chữ số sau dấu phẩy để dễ so sánh
        print(classification_report(y_test, y_pred_test, digits=5))

        results.append({
            "marker": marker, 
            "test_f1": test_f1, 
            "test_acc": test_acc,
            "params": best_params
        })

    return pd.DataFrame(results)

if __name__ == "__main__":
    run_xgb_tuned()

Loading data...

===== CHẠY EARLY PREDICTION: 10% MỐC THỜI GIAN =====


[I 2026-04-12 16:50:52,471] A new study created in memory with name: no-name-66428ec5-7902-4afa-8b2d-0c1d0b7fcd0f


Bắt đầu Tuning Optuna (Tối ưu F1 với Thr=0.5)...


[I 2026-04-12 16:51:26,485] Trial 0 finished with value: 0.7886775616238384 and parameters: {'learning_rate': 0.04031111575478693, 'max_depth': 9, 'subsample': 0.7306456073755041, 'colsample_bytree': 0.6834562298086218, 'gamma': 1.5155759067421072e-06, 'alpha': 0.00291131470975192, 'lambda': 0.02074816955390202, 'min_child_weight': 1}. Best is trial 0 with value: 0.7886775616238384.
[I 2026-04-12 16:52:30,973] Trial 1 finished with value: 0.7911342921846987 and parameters: {'learning_rate': 0.014574775657751912, 'max_depth': 9, 'subsample': 0.9223220215562218, 'colsample_bytree': 0.8249116054220035, 'gamma': 0.0005019729756084221, 'alpha': 1.778393869257412e-05, 'lambda': 0.0008776784503303316, 'min_child_weight': 10}. Best is trial 1 with value: 0.7911342921846987.
[I 2026-04-12 16:52:59,093] Trial 2 finished with value: 0.7920981722517086 and parameters: {'learning_rate': 0.03289416714720789, 'max_depth': 4, 'subsample': 0.6901678211320055, 'colsample_bytree': 0.9108316628610906, 'ga

Huấn luyện model cuối với bộ tham số tốt nhất...
KẾT QUẢ 10%: Test F1 = 0.80123 | Test Acc = 0.80120
              precision    recall  f1-score   support

           0    0.84883   0.75857   0.80117      3442
           1    0.75864   0.84888   0.80123      3077

    accuracy                        0.80120      6519
   macro avg    0.80374   0.80372   0.80120      6519
weighted avg    0.80626   0.80120   0.80119      6519


===== CHẠY EARLY PREDICTION: 40% MỐC THỜI GIAN =====


...

Huấn luyện model cuối với bộ tham số tốt nhất...
KẾT QUẢ 80%: Test F1 = 0.94272 | Test Acc = 0.94447
              precision    recall  f1-score   support

           0    0.97009   0.92330   0.94611      3442
           1    0.91859   0.96815   0.94272      3077

    accuracy                        0.94447      6519
   macro avg    0.94434   0.94573   0.94442      6519
weighted avg    0.94578   0.94447   0.94451      6519


===== CHẠY EARLY PREDICTION: 100% MỐC THỜI GIAN =====


[I 2026-04-12 20:32:23,094] A new study created in memory with name: no-name-f8cd2fc1-07ac-4474-804d-a548289b7fcc


Bắt đầu Tuning Optuna (Tối ưu F1 với Thr=0.5)...


[I 2026-04-12 20:39:56,735] Trial 0 finished with value: 0.9469622262965233 and parameters: {'learning_rate': 0.012812906133018137, 'max_depth': 3, 'subsample': 0.9578670761403844, 'colsample_bytree': 0.6141038419215369, 'gamma': 1.0871902000882278e-07, 'alpha': 2.208430571602721e-06, 'lambda': 0.8633075095375953, 'min_child_weight': 6}. Best is trial 0 with value: 0.9469622262965233.
[I 2026-04-12 20:46:31,246] Trial 1 finished with value: 0.9527897573325443 and parameters: {'learning_rate': 0.03650417033217391, 'max_depth': 8, 'subsample': 0.8550901455964252, 'colsample_bytree': 0.7286625832438787, 'gamma': 4.267826676722154e-06, 'alpha': 0.0003171941586894367, 'lambda': 0.014114734943524183, 'min_child_weight': 6}. Best is trial 1 with value: 0.9527897573325443.
[I 2026-04-12 20:51:15,074] Trial 2 finished with value: 0.9526294281933559 and parameters: {'learning_rate': 0.05389564857309426, 'max_depth': 5, 'subsample': 0.8827654205714103, 'colsample_bytree': 0.9506665033816627, 'gam

Huấn luyện model cuối với bộ tham số tốt nhất...
KẾT QUẢ 100%: Test F1 = 0.95506 | Test Acc = 0.95659
              precision    recall  f1-score   support

           0    0.97878   0.93812   0.95802      3442
           1    0.93385   0.97725   0.95506      3077

    accuracy                        0.95659      6519
   macro avg    0.95632   0.95768   0.95654      6519
weighted avg    0.95757   0.95659   0.95662      6519

